# Weighted Retention Regressions — Full Covariate Effects

This fixed notebook:

1. Runs all retention year-pair regressions in one go, including 2016–2017.
2. Removes New York / NYC where identifiable.
3. Uses CPS/MORG weights at the regression stage, preferring `weight_t`.
4. Estimates a weighted linear probability model.
5. Outputs all coefficients, not just the Illinois coefficient.
6. Reports coefficients in percentage points for race, sex, ethnicity/Hispanic status, occupation, education, industry, union status, age, earnings, and Illinois.

Interpretation: because the dependent variable is binary, `coef_pct_points = coef × 100` can be read as percentage-point differences relative to the omitted/reference category for categorical variables.


In [23]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------
PROJECT_DIR = Path.home() / "Documents" / "pdi_pensions"
DATA_DIR = PROJECT_DIR / "regression_analysis" / "analytic samples" / "retention"

# Fallback in case your folder is elsewhere
if not DATA_DIR.exists():
    alt = PROJECT_DIR / "data" / "analytic samples" / "retention"
    if alt.exists():
        DATA_DIR = alt

OUTPUT_DIR = Path.home() / "Downloads"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_PAIRS = [
    (2008, 2009),
    (2009, 2010),
    (2010, 2011),
    (2011, 2012),
    (2012, 2013),
    (2013, 2014),
    (2014, 2015),
    (2015, 2016),
    (2016, 2017),
]

FILE_PATTERN = "analytic_sample_{yy0}{yy1}_retention.csv"

DV = "stayed_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR = "pair"

# For transition outcomes t -> t+1, weight_t is preferred because the risk set is defined at t.
WEIGHT_CANDIDATES = [
    "weight_t", "wgt_t", "wtfinl_t", "finalwgt_t",
    "weight", "wgt", "wtfinl", "finalwgt",
    # earnwt is not ideal for this retention outcome, but is kept as a last-resort fallback.
    "earnwt_t", "earnwt"
]

COV_TYPE = "HC1"

EXCLUDED_STATE_FIPS = {36}
EXCLUDED_STATE_NAMES = {"ny", "new york", "new york state", "nyc", "new york city"}

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists?", DATA_DIR.exists())

if DATA_DIR.exists():
    print("CSV files found:")
    print([p.name for p in sorted(DATA_DIR.glob("*.csv"))])
else:
    print("DATA_DIR does not exist. Update DATA_DIR in this cell.")


PROJECT_DIR: /Users/vedikabaradwaj/Documents/pdi_pensions
DATA_DIR: /Users/vedikabaradwaj/Documents/pdi_pensions/regression_analysis/analytic samples/retention
DATA_DIR exists? True
CSV files found:
['analytic_sample_0809_retention.csv', 'analytic_sample_0910_retention.csv', 'analytic_sample_1011_retention.csv', 'analytic_sample_1112_retention.csv', 'analytic_sample_1213_retention.csv', 'analytic_sample_1314_retention.csv', 'analytic_sample_1415_retention.csv', 'analytic_sample_1516_retention.csv', 'analytic_sample_1617_retention.csv']


In [24]:
# -------------------------------------------------------------------
# CONTROL VARIABLE SETUP
# -------------------------------------------------------------------

# Continuous controls enter directly.
CONTINUOUS_CONTROL_CANDIDATES = [
    "age_t",
    "I(age_t**2)",
    "np.log(earnwke_t)",
]

# Categorical controls. The notebook automatically includes whichever are present.
CATEGORICAL_CONTROL_CANDIDATES = [
    # Demographics
    "C(sex_t)",
    "C(race_t)",
    "C(ethnic_t)",
    "C(ethnicity_t)",
    "C(hispan_t)",
    "C(hispanic_t)",
    "C(hisp_t)",

    # Education
    "C(grade92_t)",
    "C(educ_t)",

    # Occupation alternatives. If multiple exist, prefer docc00_t.
    "C(docc00_t)",
    "C(docc80_t)",
    "C(occ2010_t)",
    "C(occ_t)",

    # Industry alternatives. If multiple exist, prefer ind02_t.
    "C(ind02_t)",
    "C(naics2_t)",
    "C(ind_t)",

    # Union / coverage
    "C(unionmme_t)",
    "C(unioncov_t)",
]

ALL_CONTROL_CANDIDATES = CONTINUOUS_CONTROL_CANDIDATES + CATEGORICAL_CONTROL_CANDIDATES


def extract_needed_columns(terms):
    """Pull raw column names out of formula terms like C(race_t), I(age_t**2), np.log(earnwke_t)."""
    cols = set()
    for term in terms:
        for m in re.findall(r"C\(([^)]+)\)", term):
            cols.add(m.strip())
        for m in re.findall(r"np\.log\(([^)]+)\)", term):
            cols.add(m.strip())
        for expr in re.findall(r"I\(([^)]+)\)", term):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
        if not term.startswith(("C(", "I(", "np.log(")):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", term):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
    return sorted(cols)


def is_categorical_term(term):
    return term.startswith("C(")


def categorical_column_from_term(term):
    m = re.match(r"C\(([^)]+)\)", term)
    return m.group(1).strip() if m else None


def choose_one_present(terms, priority):
    present = [t for t in priority if t in terms]
    if len(present) <= 1:
        return terms
    keep = present[0]
    return [t for t in terms if (t not in priority or t == keep)]


def available_terms(df, candidate_terms):
    """Include controls only if their raw columns exist, and avoid duplicative alternatives."""
    terms = []
    for term in candidate_terms:
        raw_cols = extract_needed_columns([term])
        if all(c in df.columns for c in raw_cols):
            terms.append(term)

    # Prefer one occupation variable.
    terms = choose_one_present(terms, ["C(docc00_t)", "C(docc80_t)", "C(occ2010_t)", "C(occ_t)"])

    # Prefer one industry variable.
    terms = choose_one_present(terms, ["C(ind02_t)", "C(naics2_t)", "C(ind_t)"])

    # Prefer one ethnicity/Hispanic variable.
    terms = choose_one_present(terms, ["C(hispan_t)", "C(hispanic_t)", "C(hisp_t)", "C(ethnic_t)", "C(ethnicity_t)"])

    return terms


def build_formula(dv, controls):
    rhs_terms = [TREATMENT_VAR] + controls
    return f"{dv} ~ " + " + ".join(rhs_terms)


def detect_weight_var(df):
    for c in WEIGHT_CANDIDATES:
        if c in df.columns:
            return c
    return None


In [25]:
# -------------------------------------------------------------------
# FILE LOADING + NY/NYC REMOVAL
# -------------------------------------------------------------------

def pair_to_file(y0, y1):
    yy0 = str(y0)[-2:]
    yy1 = str(y1)[-2:]
    return DATA_DIR / FILE_PATTERN.format(y0=y0, y1=y1, yy0=yy0, yy1=yy1)


def remove_ny_nyc(df):
    """
    Remove New York / NYC when possible.
    If only state-level CPS identifiers exist, this removes New York State, FIPS 36.
    In merged files, stfips may be shared across t and t+1 because it was used as a merge key.
    """
    before = len(df)
    mask = pd.Series(False, index=df.index)
    used_cols = []

    # Numeric state FIPS columns.
    for col in ["stfips_t", "stfips_t1", "statefip_t", "state_fips_t", "state_t", "stfips"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.isin(EXCLUDED_STATE_FIPS)
            used_cols.append(col)

    # String state/location columns.
    for col in ["state_name_t", "state_abbrev_t", "state_t", "city_t", "metro_t", "cbsatitle_t"]:
        if col in df.columns:
            vals = df[col].astype("object").astype(str).str.strip().str.lower()
            mask = mask | vals.isin(EXCLUDED_STATE_NAMES) | vals.str.contains("new york", na=False)
            used_cols.append(col)

    # NYC CBSA code if present: 35620 = New York-Newark-Jersey City metro.
    for col in ["cbsafips_t", "cbsa_t", "metfips_t"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.eq(35620)
            used_cols.append(col)

    df2 = df.loc[~mask].copy()
    dropped = before - len(df2)
    print(f"Removed NY/NYC rows: {dropped:,} using columns {sorted(set(used_cols)) if used_cols else 'none found'}")
    return df2


def load_pair(y0, y1):
    path = pair_to_file(y0, y1)
    print("\n" + "=" * 90)
    print(f"PAIR {y0}-{y1}")
    print("Looking for:", path)

    if not path.exists():
        print(f"WARNING: file not found, skipping: {path.name}")
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    df[PAIR_VAR] = f"{str(y0)[-2:]}{str(y1)[-2:]}"

    print("Loaded shape:", df.shape)
    df = remove_ny_nyc(df)
    print("Shape after NY/NYC removal:", df.shape)
    print("Detected weight variable:", detect_weight_var(df))
    return df


In [26]:
# -------------------------------------------------------------------
# DATA CLEANING FOR MODEL
# -------------------------------------------------------------------

def clean_categorical_series(s):
    """
    Convert to plain Python object strings.
    This avoids statsmodels/Patsy errors like:
    Cannot interpret 'string[python]' as a data type.
    """
    out = s.astype("object")
    out = out.where(~pd.isna(out), "MISSING")
    out = out.astype(str).str.strip()
    out = out.replace({
        "": "MISSING",
        "<NA>": "MISSING",
        "nan": "MISSING",
        "NaN": "MISSING",
        "None": "MISSING",
        "none": "MISSING",
    })
    return out.astype("object")


def prepare_model_data(df, controls):
    weight_var = detect_weight_var(df)
    if weight_var is None:
        return None, None, "No weight variable found"

    missing = [c for c in [DV, TREATMENT_VAR] if c not in df.columns]
    if missing:
        return None, None, f"Missing required columns: {missing}"

    data = df.copy()

    # Required numeric variables.
    data[weight_var] = pd.to_numeric(data[weight_var], errors="coerce")
    data[DV] = pd.to_numeric(data[DV], errors="coerce")
    data[TREATMENT_VAR] = pd.to_numeric(data[TREATMENT_VAR], errors="coerce")

    # Continuous controls.
    numeric_cols = [DV, TREATMENT_VAR, weight_var]
    for term in controls:
        if not is_categorical_term(term):
            numeric_cols.extend(extract_needed_columns([term]))
    numeric_cols = sorted(set(c for c in numeric_cols if c in data.columns))

    for c in numeric_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    # Categorical controls: fill missing rather than dropping rows.
    categorical_cols = []
    for term in controls:
        if is_categorical_term(term):
            c = categorical_column_from_term(term)
            if c in data.columns:
                categorical_cols.append(c)

    for c in categorical_cols:
        data[c] = clean_categorical_series(data[c])

    # Log earnings requires positive earnings.
    if "np.log(earnwke_t)" in controls and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    # Drop missing only for numeric variables required by WLS.
    before = len(data)
    data = data.dropna(subset=numeric_cols).copy()
    data = data[data[weight_var] > 0].copy()
    after = len(data)

    if data.empty:
        return None, weight_var, "No rows left after cleaning"

    return data, weight_var, None


In [27]:
# -------------------------------------------------------------------
# REGRESSION FUNCTION: RETURN ALL COEFFICIENTS, NOT JUST ILLINOIS
# -------------------------------------------------------------------

def classify_term(term):
    term_l = term.lower()

    if term == "Intercept":
        return "intercept"
    if term == TREATMENT_VAR:
        return "treatment"
    if "race" in term_l:
        return "race"
    if "sex" in term_l:
        return "sex"
    if any(x in term_l for x in ["hisp", "ethnic", "ethnicity"]):
        return "ethnicity_hispanic"
    if any(x in term_l for x in ["docc", "occ"]):
        return "occupation"
    if "grade" in term_l or "educ" in term_l:
        return "education"
    if "ind" in term_l or "naics" in term_l:
        return "industry"
    if "union" in term_l:
        return "union"
    if "age" in term_l:
        return "age"
    if "earn" in term_l or "log" in term_l:
        return "earnings"
    return "other"


def run_weighted_lpm_all_terms(df, y0, y1):
    controls = available_terms(df, ALL_CONTROL_CANDIDATES)
    formula = build_formula(DV, controls)

    data, weight_var, error = prepare_model_data(df, controls)
    if error:
        print(f"Skipping {y0}-{y1}: {error}")
        return []

    print(f"\nRunning weighted full-covariate LPM for {y0}-{y1}")
    print("Formula:", formula)
    print(f"Rows used: {len(data):,} / {len(df):,}")
    print("Weight variable:", weight_var)
    print("Controls included:", controls)

    try:
        result = smf.wls(
            formula=formula,
            data=data,
            weights=data[weight_var]
        ).fit(cov_type=COV_TYPE)
    except Exception as e:
        print(f"Regression failed for {y0}-{y1}: {e}")
        return []

    conf_int = result.conf_int()
    rows = []

    for term in result.params.index:
        coef = result.params.get(term, np.nan)
        se = result.bse.get(term, np.nan)
        pval = result.pvalues.get(term, np.nan)
        ci_low = conf_int.loc[term, 0] if term in conf_int.index else np.nan
        ci_high = conf_int.loc[term, 1] if term in conf_int.index else np.nan

        rows.append({
            "pair": f"{y0}-{y1}",
            "pair_code": f"{str(y0)[-2:]}{str(y1)[-2:]}",
            "dv": DV,
            "term": term,
            "term_group": classify_term(term),
            "coef": coef,
            "coef_pct_points": coef * 100 if pd.notna(coef) else np.nan,
            "std_err": se,
            "std_err_pct_points": se * 100 if pd.notna(se) else np.nan,
            "p_value": pval,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "ci_low_pct_points": ci_low * 100 if pd.notna(ci_low) else np.nan,
            "ci_high_pct_points": ci_high * 100 if pd.notna(ci_high) else np.nan,
            "nobs": int(result.nobs),
            "r_squared": result.rsquared,
            "weight_var": weight_var,
            "controls": ", ".join(controls),
            "file": pair_to_file(y0, y1).name,
        })

    return rows


In [28]:
# -------------------------------------------------------------------
# RUN ALL YEAR-PAIR REGRESSIONS
# -------------------------------------------------------------------

all_coef_rows = []
loaded_pairs = {}

for y0, y1 in YEAR_PAIRS:
    df_pair = load_pair(y0, y1)
    if df_pair is None:
        continue

    loaded_pairs[f"{y0}-{y1}"] = df_pair
    rows = run_weighted_lpm_all_terms(df_pair, y0, y1)
    all_coef_rows.extend(rows)

all_results_table = pd.DataFrame(all_coef_rows)

print("\n" + "=" * 90)
print("DONE")
print(f"Coefficient rows produced: {len(all_results_table):,}")
print(f"Year pairs successfully loaded: {list(loaded_pairs.keys())}")

display(all_results_table.head(20))



PAIR 2008-2009
Looking for: /Users/vedikabaradwaj/Documents/pdi_pensions/regression_analysis/analytic samples/retention/analytic_sample_0809_retention.csv
Loaded shape: (1686, 207)
Removed NY/NYC rows: 686 using columns ['cbsafips_t', 'state_t', 'stfips_t', 'stfips_t1']
Shape after NY/NYC removal: (1000, 207)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2008-2009
Formula: stayed_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(unionmme_t) + C(unioncov_t)
Rows used: 930 / 1,000
Weight variable: weight_t
Controls included: ['age_t', 'I(age_t**2)', 'np.log(earnwke_t)', 'C(sex_t)', 'C(race_t)', 'C(ethnic_t)', 'C(grade92_t)', 'C(docc00_t)', 'C(ind02_t)', 'C(unionmme_t)', 'C(unioncov_t)']

PAIR 2009-2010
Looking for: /Users/vedikabaradwaj/Documents/pdi_pensions/regression_analysis/analytic samples/retention/analytic_sample_0910_retention.csv
Loaded shape: (1746, 

,pair,pair_code,dv,term,term_group,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low,ci_high,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,controls,file
0,2008-2009,0809,stayed_state_local,Intercept,intercept,0.065537,6.553750,0.301165,30.116505,8.277305e-01,-0.524735,0.655810,-52.473516,65.581015,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
1,2008-2009,0809,stayed_state_local,C(sex_t)[T.2],sex,0.052328,5.232763,0.041890,4.188997,2.116035e-01,-0.029775,0.134430,-2.977520,13.443047,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
2,2008-2009,0809,stayed_state_local,C(race_t)[T.10],race,0.096721,9.672113,0.177089,17.708922,5.849479e-01,-0.250367,0.443810,-25.036737,44.380963,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
3,2008-2009,0809,stayed_state_local,C(race_t)[T.2],race,0.058269,5.826879,0.057383,5.738317,3.098993e-01,-0.054200,0.170738,-5.420017,17.073774,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
4,2008-2009,0809,stayed_state_local,C(race_t)[T.3],race,-0.245236,-24.523643,0.373832,37.383185,5.118194e-01,-0.977933,0.487461,-97.793339,48.746052,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
5,2008-2009,0809,stayed_state_local,C(race_t)[T.4],race,-0.178345,-17.834511,0.168427,16.842694,2.896512e-01,-0.508456,0.151766,-50.845584,15.176563,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
6,2008-2009,0809,stayed_state_local,C(race_t)[T.5],race,0.371796,37.179560,0.056555,5.655511,4.896783e-11,0.260950,0.482642,26.094962,48.264158,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
7,2008-2009,0809,stayed_state_local,C(race_t)[T.7],race,0.362308,36.230769,0.190923,19.092291,5.774024e-02,-0.011894,0.736510,-1.189433,73.650972,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
8,2008-2009,0809,stayed_state_local,C(ethnic_t)[T.2.0],ethnicity_hispanic,-0.045227,-4.522725,0.207188,20.718778,8.272023e-01,-0.451308,0.360853,-45.130784,36.085334,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv
9,2008-2009,0809,stayed_state_local,C(ethnic_t)[T.3.0],ethnicity_hispanic,0.128512,12.851163,0.182173,18.217263,4.805372e-01,-0.228540,0.485563,-22.854017,48.556344,930,0.126703,weight_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_retention.csv


In [29]:
# -------------------------------------------------------------------
# TABLE 1: ILLINOIS EFFECT BY YEAR PAIR
# -------------------------------------------------------------------

if all_results_table.empty:
    print("No results produced. Check DATA_DIR, FILE_PATTERN, required variables, and weight variables.")
    illinois_effect_table = pd.DataFrame()
else:
    illinois_effect_table = (
        all_results_table
        .query("term_group == 'treatment'")
        [["pair", "pair_code", "term", "coef", "coef_pct_points", "std_err", "std_err_pct_points",
          "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var", "file"]]
        .sort_values("pair_code")
        .reset_index(drop=True)
    )

display(illinois_effect_table)


,pair,pair_code,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,file
0,2008-2009,0809,illinois,0.035081,3.508077,0.036504,3.650408,0.336547,-3.646590,10.662745,930,0.126703,weight_t,analytic_sample_0809_retention.csv
1,2009-2010,0910,illinois,0.057581,5.758071,0.035826,3.582554,0.107999,-1.263606,12.779749,934,0.097802,weight_t,analytic_sample_0910_retention.csv
2,2010-2011,1011,illinois,0.003852,0.385200,0.037254,3.725436,0.917648,-6.916520,7.686920,866,0.125669,weight_t,analytic_sample_1011_retention.csv
3,2011-2012,1112,illinois,0.001188,0.118840,0.038599,3.859852,0.975438,-7.446331,7.684012,803,0.124615,weight_t,analytic_sample_1112_retention.csv
4,2012-2013,1213,illinois,0.035833,3.583282,0.038854,3.885419,0.356404,-4.031999,11.198563,754,0.127503,weight_t,analytic_sample_1213_retention.csv
5,2013-2014,1314,illinois,0.068344,6.834364,0.039201,3.920103,0.081261,-0.848896,14.517624,660,0.253108,weight_t,analytic_sample_1314_retention.csv
6,2014-2015,1415,illinois,-0.020244,-2.024411,0.038066,3.806623,0.594856,-9.485254,5.436433,500,0.277939,weight_t,analytic_sample_1415_retention.csv
7,2015-2016,1516,illinois,-0.073214,-7.321380,0.039753,3.975314,0.065517,-15.112853,0.470093,568,0.281659,weight_t,analytic_sample_1516_retention.csv
8,2016-2017,1617,illinois,-0.010742,-1.074205,0.041399,4.139869,0.795266,-9.188198,7.039789,529,0.248859,weight_t,analytic_sample_1617_retention.csv


In [30]:
# -------------------------------------------------------------------
# TABLE 2: ALL COVARIATE EFFECTS BY YEAR PAIR
# -------------------------------------------------------------------

if all_results_table.empty:
    print("No results produced.")
    all_covariate_effects_table = pd.DataFrame()
else:
    all_covariate_effects_table = (
        all_results_table
        .query("term != 'Intercept'")
        [["pair", "pair_code", "term_group", "term", "coef", "coef_pct_points", "std_err",
          "std_err_pct_points", "p_value", "ci_low_pct_points", "ci_high_pct_points",
          "nobs", "r_squared", "weight_var"]]
        .sort_values(["pair_code", "term_group", "term"])
        .reset_index(drop=True)
    )

display(all_covariate_effects_table)


,pair,pair_code,term_group,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.000027,-0.002704,0.000089,0.008882,0.760767,-0.020113,0.014704,930,0.126703,weight_t
1,2008-2009,0809,age,age_t,0.003137,0.313674,0.008497,0.849652,0.711994,-1.351614,1.978962,930,0.126703,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),0.034387,3.438662,0.023225,2.322478,0.138713,-1.113311,7.990634,930,0.126703,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.33],-0.301234,-30.123438,0.271842,27.184238,0.267809,-83.403566,23.156690,930,0.126703,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.35],-0.677184,-67.718416,0.380258,38.025809,0.074936,-142.247633,6.810801,930,0.126703,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,2016-2017,1617,sex,C(sex_t)[T.2],-0.018852,-1.885232,0.050184,5.018378,0.707166,-11.721071,7.950608,529,0.248859,weight_t
818,2016-2017,1617,treatment,illinois,-0.010742,-1.074205,0.041399,4.139869,0.795266,-9.188198,7.039789,529,0.248859,weight_t
819,2016-2017,1617,union,C(unioncov_t)[T.2.0],-0.181920,-18.191976,0.134176,13.417604,0.175154,-44.489996,8.106043,529,0.248859,weight_t
820,2016-2017,1617,union,C(unioncov_t)[T.MISSING],0.256095,25.609456,0.107331,10.733135,0.017032,4.572898,46.646014,529,0.248859,weight_t


In [31]:
# -------------------------------------------------------------------
# TABLE 3: DEMOGRAPHIC, OCCUPATION, INDUSTRY, UNION, AGE, EARNINGS EFFECTS
# -------------------------------------------------------------------

focus_groups = [
    "treatment", "race", "sex", "ethnicity_hispanic",
    "occupation", "education", "industry", "union", "age", "earnings"
]

if all_results_table.empty:
    print("No results produced.")
    focus_effects_table = pd.DataFrame()
else:
    focus_effects_table = (
        all_results_table
        .query("term_group in @focus_groups and term != 'Intercept'")
        [["pair", "pair_code", "term_group", "term", "coef_pct_points", "std_err_pct_points",
          "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var"]]
        .sort_values(["pair_code", "term_group", "term"])
        .reset_index(drop=True)
    )

display(focus_effects_table)


,pair,pair_code,term_group,term,coef_pct_points,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.002704,0.008882,0.760767,-0.020113,0.014704,930,0.126703,weight_t
1,2008-2009,0809,age,age_t,0.313674,0.849652,0.711994,-1.351614,1.978962,930,0.126703,weight_t
2,2008-2009,0809,earnings,np.log(earnwke_t),3.438662,2.322478,0.138713,-1.113311,7.990634,930,0.126703,weight_t
3,2008-2009,0809,education,C(grade92_t)[T.33],-30.123438,27.184238,0.267809,-83.403566,23.156690,930,0.126703,weight_t
4,2008-2009,0809,education,C(grade92_t)[T.35],-67.718416,38.025809,0.074936,-142.247633,6.810801,930,0.126703,weight_t
...,...,...,...,...,...,...,...,...,...,...,...,...
817,2016-2017,1617,sex,C(sex_t)[T.2],-1.885232,5.018378,0.707166,-11.721071,7.950608,529,0.248859,weight_t
818,2016-2017,1617,treatment,illinois,-1.074205,4.139869,0.795266,-9.188198,7.039789,529,0.248859,weight_t
819,2016-2017,1617,union,C(unioncov_t)[T.2.0],-18.191976,13.417604,0.175154,-44.489996,8.106043,529,0.248859,weight_t
820,2016-2017,1617,union,C(unioncov_t)[T.MISSING],25.609456,10.733135,0.017032,4.572898,46.646014,529,0.248859,weight_t


In [32]:
# -------------------------------------------------------------------
# TABLE 4: AVERAGE COEFFICIENT ACROSS YEAR PAIRS BY TERM
# -------------------------------------------------------------------

if all_results_table.empty:
    print("No results produced.")
    average_effects_by_term = pd.DataFrame()
else:
    average_effects_by_term = (
        all_results_table
        .query("term != 'Intercept'")
        .groupby(["term_group", "term"], as_index=False)
        .agg(
            mean_coef=("coef", "mean"),
            mean_coef_pct_points=("coef_pct_points", "mean"),
            median_coef_pct_points=("coef_pct_points", "median"),
            num_year_pairs=("pair_code", "nunique"),
            mean_nobs=("nobs", "mean"),
        )
        .sort_values(["term_group", "term"])
        .reset_index(drop=True)
    )

display(average_effects_by_term)


,term_group,term,mean_coef,mean_coef_pct_points,median_coef_pct_points,num_year_pairs,mean_nobs
0,age,I(age_t ** 2),-0.000070,-0.007049,-0.004900,9,727.111111
1,age,age_t,0.007047,0.704654,0.313674,9,727.111111
2,earnings,np.log(earnwke_t),0.023728,2.372820,1.301506,9,727.111111
3,education,C(grade92_t)[T.33],-0.450404,-45.040401,-45.040401,2,932.000000
4,education,C(grade92_t)[T.34],0.152470,15.247049,14.694629,6,691.833333
...,...,...,...,...,...,...,...
134,sex,C(sex_t)[T.2],-0.005702,-0.570215,-0.747003,9,727.111111
135,treatment,illinois,0.010853,1.085316,0.385200,9,727.111111
136,union,C(unioncov_t)[T.2.0],-0.117101,-11.710138,-10.305285,9,727.111111
137,union,C(unioncov_t)[T.MISSING],0.145124,14.512450,8.812277,9,727.111111


In [33]:
# -------------------------------------------------------------------
# SAVE TABLES
# -------------------------------------------------------------------

if not all_results_table.empty:
    out_all_csv = OUTPUT_DIR / "retention_weighted_all_covariate_coefficients.csv"
    out_illinois_csv = OUTPUT_DIR / "retention_weighted_illinois_effects.csv"
    out_focus_csv = OUTPUT_DIR / "retention_weighted_demographic_occupation_effects.csv"
    out_avg_csv = OUTPUT_DIR / "retention_weighted_average_effects_by_term.csv"
    out_xlsx = OUTPUT_DIR / "retention_weighted_regression_tables.xlsx"

    all_results_table.to_csv(out_all_csv, index=False)
    illinois_effect_table.to_csv(out_illinois_csv, index=False)
    focus_effects_table.to_csv(out_focus_csv, index=False)
    average_effects_by_term.to_csv(out_avg_csv, index=False)

    with pd.ExcelWriter(out_xlsx) as writer:
        illinois_effect_table.to_excel(writer, sheet_name="illinois_effects", index=False)
        all_covariate_effects_table.to_excel(writer, sheet_name="all_covariates", index=False)
        focus_effects_table.to_excel(writer, sheet_name="demo_occ_effects", index=False)
        average_effects_by_term.to_excel(writer, sheet_name="avg_by_term", index=False)
        all_results_table.to_excel(writer, sheet_name="raw_all_terms", index=False)

    print("Saved:")
    print("-", out_all_csv)
    print("-", out_illinois_csv)
    print("-", out_focus_csv)
    print("-", out_avg_csv)
    print("-", out_xlsx)
else:
    print("No results to save.")


Saved:
- /Users/vedikabaradwaj/Downloads/retention_weighted_all_covariate_coefficients.csv
- /Users/vedikabaradwaj/Downloads/retention_weighted_illinois_effects.csv
- /Users/vedikabaradwaj/Downloads/retention_weighted_demographic_occupation_effects.csv
- /Users/vedikabaradwaj/Downloads/retention_weighted_average_effects_by_term.csv
- /Users/vedikabaradwaj/Downloads/retention_weighted_regression_tables.xlsx


## Notes on interpreting the output

- These are weighted linear probability models using the detected CPS/MORG weight, preferably `weight_t`.
- For a transition outcome from year `t` to `t+1`, `weight_t` is the right default because the risk set is defined at `t`.
- `coef_pct_points` is the coefficient multiplied by 100.
- For categorical variables, coefficients are relative to the omitted/reference category chosen by Patsy/statsmodels.
- If occupation or industry has many categories, the tables will be long. That is expected.
- Missing categorical covariates are coded as `MISSING` to avoid collapsing the sample.
- Continuous missing values are still dropped because WLS cannot estimate with missing numeric values.
